# 실습 6: LLM on Databricks — AI 양념 🌶️

**목표**: Databricks에서 LLM을 저비용으로 활용하는 방법을 맛보기

**3파트 구성**:
- Part A: Foundation Model API 기본 호출
- Part B: AI Functions — SQL로 AI 하기
- Part C: 간단 RAG 체험 (Vector Search + LLM)

**소요 시간**: ~50분

⚠️ **비용 주의**: 토큰당 과금! `max_tokens=256`으로 제한합니다.

## ⚙️ 환경 설정

In [0]:
# 🔧 수강생 환경에 맞게 수정하세요
CATALOG = "3dt005_databricks"      # 예: "main", "jhleews"
SCHEMA = "wine"          # 예: "default", "wine"

# LLM 모델 엔드포인트 (사용 가능한 모델로 변경)
LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"

print(f"📌 카탈로그: {CATALOG}.{SCHEMA}")
print(f"📌 LLM 모델: {LLM_MODEL}")

📌 카탈로그: 3dt005_databricks.wine
📌 LLM 모델: databricks-meta-llama-3-3-70b-instruct


## Part A: Foundation Model API 기본 호출 (15분)

### Databricks Foundation Model API란?
- Databricks가 호스팅하는 LLM (DBRX, Llama 3, Mixtral 등)
- 별도 배포 없이 API 호출만으로 사용
- 토큰당 과금 — 실습에 적합한 저비용 구조

💡 **사용 가능한 모델 확인**: 왼쪽 메뉴 > Serving > Foundation Model APIs 탭

In [0]:
%pip install databricks-vectorsearch
# ↑ ModuleNotFoundError 발생 시 이 줄의 주석을 해제하고 실행한 뒤

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow.deployments

# Databricks Foundation Model API 클라이언트
client = mlflow.deployments.get_deploy_client("databricks")

# LLM에 질문하기
response = client.predict(
    endpoint=LLM_MODEL,
    inputs={
        "messages": [
            {"role": "system", "content": "You are a helpful data science assistant. Answer in Korean."},
            {"role": "user", "content": "머신러닝에서 과적합(overfitting)이 무엇이고, 어떻게 방지하나요? 간단히 설명해주세요."}
        ],
        "max_tokens": 256,
        "temperature": 0.7,
    }
)

print("🤖 LLM 응답:")
print(response["choices"][0]["message"]["content"])

🤖 LLM 응답:
머신러닝에서 과적합(overfitting)은 모델이 훈련 데이터에 너무 잘 맞추어져서 새로운 데이터에 적용할 때 성능이 낮아지는 현상입니다. 이는 모델이 훈련 데이터의 노이즈나 이상값에 너무 민감하게 반응하여 일반화가 안 되는 경우에 발생합니다.

과적합을 방지하는 방법으로는 다음과 같은 것들이 있습니다:

1. **정규화(Regularization)**: 모델의 복잡도를 줄이는 방법으로, 모델의 가중치를 조정하여 과적합을 방지합니다.
2. **드롭아웃(Dropout)**: 모델의 일부 뉴런을 무작위로 삭제하여 과적합을 방지합니다.
3. **데이터 증강(Data Augmentation)**: 훈련 데이터를 증강하여 모델이 다양한 데이터에 잘 일반화하도록 합니다.
4. **교차 검증(Cross-Validation)**: 모델을 여러 번 훈련하여 과적합을 방지합니다.
5. **모델의 복잡도 줄이기**: 모델의 복잡도를 줄여 과적합을 방지합니다.

이러한 방법들을 통해 과


### 프롬프트 엔지니어링 맛보기
같은 질문, 다른 프롬프트 → 다른 결과

In [0]:
# Few-shot 프롬프트 예시: 와인 설명 생성
response = client.predict(
    endpoint=LLM_MODEL,
    inputs={
        "messages": [
            {"role": "system", "content": """You are a sommelier. Given wine chemical properties, write a tasting note.

Examples:
- Alcohol 12.5%, Residual sugar 5g → "Medium-bodied dry wine. Clean finish."
- Alcohol 9.0%, Residual sugar 45g → "Light-bodied sweet wine. Rich fruit aroma."
"""},
            {"role": "user", "content": "Alcohol 11.8%, Residual sugar 1.2g, Acidity 0.32, pH 3.18 — Describe this wine."}
        ],
        "max_tokens": 128,
    }
)

print("🍷 Sommelier AI:")
print(response["choices"][0]["message"]["content"])

🍷 Sommelier AI:
"Crisp and refreshing wine with a light to medium body. The low residual sugar indicates a dry style, while the moderate acidity and typical pH level suggest a well-balanced wine with a lively character. The finish is likely to be clean and refreshing, making it a great accompaniment to a variety of dishes."


## Part B: AI Functions — SQL로 AI 하기 (20분)

### ai_query() — SQL 한 줄로 LLM 호출!
💡 **Databricks만의 킬러 기능**: Python 몰라도 SQL만으로 AI를 데이터 파이프라인에 통합

In [0]:
# 샘플 고객 리뷰 데이터 생성
from pyspark.sql import Row

reviews = spark.createDataFrame([
    Row(review_id=1, text="This wine is absolutely fantastic! Best purchase ever.", rating=5),
    Row(review_id=2, text="Terrible. Cork taint ruined the entire bottle. Waste of money.", rating=1),
    Row(review_id=3, text="Decent for the price. Nothing special but drinkable.", rating=3),
    Row(review_id=4, text="Wonderful aroma and balanced flavors. Would buy again!", rating=5),
    Row(review_id=5, text="Too acidic for my taste. Not recommended.", rating=2),
])

reviews.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.wine_reviews_lab")
print(f"✅ 리뷰 테이블 생성: {CATALOG}.{SCHEMA}.wine_reviews_lab")

✅ 리뷰 테이블 생성: 3dt005_databricks.wine.wine_reviews_lab


In [0]:
# 💡 ai_query()로 감성 분석 — SQL 한 줄! (spark.sql로 변수 참조)
display(spark.sql(f"""
  SELECT
    review_id,
    text,
    rating,
    ai_query(
      '{LLM_MODEL}',
      CONCAT('Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL. Reply with ONLY the classification word. Review: ', text)
    ) AS ai_sentiment
  FROM {CATALOG}.{SCHEMA}.wine_reviews_lab
"""))

review_id,text,rating,ai_sentiment
4,Wonderful aroma and balanced flavors. Would buy again!,5,POSITIVE
5,Too acidic for my taste. Not recommended.,2,NEGATIVE
2,Terrible. Cork taint ruined the entire bottle. Waste of money.,1,NEGATIVE
1,This wine is absolutely fantastic! Best purchase ever.,5,POSITIVE
3,Decent for the price. Nothing special but drinkable.,3,NEUTRAL


In [0]:
# 카테고리 자동 분류
display(spark.sql(f"""
  SELECT
    review_id,
    text,
    ai_query(
      '{LLM_MODEL}',
      CONCAT('Classify this wine review into ONE category: TASTE, AROMA, VALUE, QUALITY, OTHER. Reply with only the category. Review: ', text)
    ) AS category
  FROM {CATALOG}.{SCHEMA}.wine_reviews_lab
"""))

review_id,text,category
4,Wonderful aroma and balanced flavors. Would buy again!,QUALITY
5,Too acidic for my taste. Not recommended.,TASTE
2,Terrible. Cork taint ruined the entire bottle. Waste of money.,QUALITY
1,This wine is absolutely fantastic! Best purchase ever.,QUALITY
3,Decent for the price. Nothing special but drinkable.,VALUE


## Part C: 간단 RAG 체험 (25분)

### RAG(Retrieval-Augmented Generation)란?
- LLM은 학습 데이터에 없는 내부 문서를 모릅니다
- **Vector Search**로 관련 문서를 찾고 → LLM에게 전달하여 답변 생성
- 사내 문서 기반 Q&A, 고객 지원 챗봇 등에 활용

```
[사용자 질문]
     ↓
[Vector Search] → 관련 문서 2~3개 검색
     ↓
[LLM] ← 질문 + 검색된 문서 (컨텍스트)
     ↓
[답변 생성]
```

### Step 1: 지식 문서 테이블 생성

In [0]:
# 소규모 와인 지식 문서 생성
wine_docs = spark.createDataFrame([
    Row(doc_id=1, content="Chardonnay is a white wine grape variety with buttery, oaky characteristics. In cool climates it shows apple and citrus flavors, while warm climates bring tropical fruit notes."),
    Row(doc_id=2, content="Cabernet Sauvignon is the flagship full-bodied red wine. It features blackcurrant, cedar, and vanilla aromas with high tannin and acidity."),
    Row(doc_id=3, content="Wine acidity is crucial for freshness and balance. A pH of 3.0-3.4 is typical, with lower values indicating more tartness. The balance between acidity and residual sugar is key to taste."),
    Row(doc_id=4, content="Wine pairing basics: Red wines pair with red meat, white wines with fish and chicken. Sweet wines go with desserts, but the wine should be sweeter than the food."),
    Row(doc_id=5, content="In MLOps, model monitoring detects data drift and model performance degradation. Databricks Lakehouse Monitoring provides automatic drift detection using statistical tests."),
])

wine_docs.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.wine_knowledge_lab")
print(f"✅ 지식 문서 테이블 생성: {CATALOG}.{SCHEMA}.wine_knowledge_lab")

✅ 지식 문서 테이블 생성: 3dt005_databricks.wine.wine_knowledge_lab


### Step 2: Change Data Feed 활성화
Vector Search의 Delta Sync 인덱스는 테이블 변경 추적이 필요합니다.

In [0]:
spark.sql(f"""
  ALTER TABLE {CATALOG}.{SCHEMA}.wine_knowledge_lab
  SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print("✅ Change Data Feed 활성화 완료")

✅ Change Data Feed 활성화 완료


### Step 3: Vector Search 엔드포인트 확인 & 인덱스 생성

⚠️ **Vector Search 엔드포인트는 사전에 생성되어 있어야 합니다** (5~10분 소요)

엔드포인트가 없으면 아래 코드로 생성할 수 있습니다:
```python
vsc.create_endpoint(name="my-endpoint", endpoint_type="STANDARD")
```
또는 UI: 왼쪽 Compute → Vector Search 탭 → Create

In [0]:
vsc.create_endpoint(name="my-endpoint", endpoint_type="STANDARD")

{'name': 'my-endpoint',
 'creator': '3dt005@msacademy.msai.kr',
 'creation_timestamp': 1774336011881,
 'last_updated_timestamp': 1774336011881,
 'endpoint_type': 'STANDARD',
 'last_updated_user': '3dt005@msacademy.msai.kr',
 'id': '75716914-b557-4292-9054-fe9851d1c233',
 'endpoint_status': {'state': 'ONLINE'},
 'num_indexes': 0}

In [0]:
# dbutils.library.restartPython() # 실행 → 환경 설정 셀부터 다시 실행

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient(disable_notice=True)

# ⚠️ 강사가 사전에 생성해둔 엔드포인트 이름으로 수정하세요
VS_ENDPOINT = "my-endpoint"   # 수정 필요

# 엔드포인트 상태 확인
try:
    ep_info = vsc.get_endpoint(VS_ENDPOINT)
    print(f"✅ 엔드포인트 '{VS_ENDPOINT}' 확인됨")
except Exception as e:
    print(f"❌ 엔드포인트를 찾을 수 없습니다: {e}")
    print("→ VS_ENDPOINT 이름을 확인하거나, 새로 생성해주세요.")

✅ 엔드포인트 'my-endpoint' 확인됨


In [0]:
# 인덱스 생성
INDEX_NAME = f"{CATALOG}.{SCHEMA}.wine_knowledge_index"

try:
    vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=f"{CATALOG}.{SCHEMA}.wine_knowledge_lab",
        pipeline_type="TRIGGERED",
        primary_key="doc_id",
        embedding_source_column="content",
        embedding_model_endpoint_name="databricks-bge-large-en",
    )
    print(f"✅ Vector Search 인덱스 생성 시작: {INDEX_NAME}")
    print("⏳ 인덱스 준비까지 5~10분 소요됩니다.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"ℹ️ 인덱스가 이미 존재합니다: {INDEX_NAME}")
    else:
        print(f"⚠️ {e}")

✅ Vector Search 인덱스 생성 시작: 3dt005_databricks.wine.wine_knowledge_index
⏳ 인덱스 준비까지 5~10분 소요됩니다.


In [0]:
# 인덱스 상태 확인 (ready: True 가 될 때까지 이 셀을 반복 실행)
index_info = vsc.get_index(VS_ENDPOINT, INDEX_NAME).describe()
status = index_info.get("status", {})
print(f"상태: {status.get('detailed_state', 'UNKNOWN')}")
print(f"준비 완료: {status.get('ready', False)}")

if not status.get("ready", False):
    print("\n⏳ 아직 준비 중입니다. 1~2분 후 이 셀을 다시 실행해주세요.")
else:
    print("\n✅ 인덱스가 준비되었습니다! 다음 셀로 진행하세요.")

상태: ONLINE_NO_PENDING_UPDATE
준비 완료: True

✅ 인덱스가 준비되었습니다! 다음 셀로 진행하세요.


### Step 4: 유사 문서 검색 테스트

In [0]:
# 유사 문서 검색 테스트
results = vsc.get_index(
    endpoint_name=VS_ENDPOINT,
    index_name=INDEX_NAME,
).similarity_search(
    query_text="What wine pairs well with steak?",
    columns=["doc_id", "content"],
    num_results=2,
)

print("🔍 Search results:")
for doc in results.get("result", {}).get("data_array", []):
    print(f"  - [doc_id={doc[0]}] {doc[1][:100]}...")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔍 Search results:
  - [doc_id=4.0] Wine pairing basics: Red wines pair with red meat, white wines with fish and chicken. Sweet wines go...
  - [doc_id=2.0] Cabernet Sauvignon is the flagship full-bodied red wine. It features blackcurrant, cedar, and vanill...


### Step 5: RAG 파이프라인 — 검색 + LLM 답변 생성

In [0]:
def rag_answer(question):
    """Simple RAG: Search → Build context → LLM answer"""

    # 1️⃣ Vector Search로 관련 문서 검색
    search_results = vsc.get_index(VS_ENDPOINT, INDEX_NAME).similarity_search(
        query_text=question,
        columns=["content"],
        num_results=2,
    )

    # 2️⃣ 검색된 문서를 컨텍스트로 구성
    context_docs = search_results.get("result", {}).get("data_array", [])
    context = "\n".join([doc[0] for doc in context_docs])

    # 3️⃣ LLM에게 질문 + 컨텍스트 전달
    response = client.predict(
        endpoint=LLM_MODEL,
        inputs={
            "messages": [
                {"role": "system", "content": f"""You are a wine expert. Answer the question based ONLY on the reference documents below.
Answer concisely. If the information is not in the documents, say "Information not found in documents."

Reference documents:
{context}"""},
                {"role": "user", "content": question}
            ],
            "max_tokens": 256,
        }
    )

    return response["choices"][0]["message"]["content"]

In [0]:
# RAG 테스트
questions = [
    "What are the characteristics of Chardonnay wine?",
    "What wine pairs well with steak?",
    "How does Databricks detect model drift?",
]

for q in questions:
    print(f"\n❓ {q}")
    print(f"💬 {rag_answer(q)}")
    print("-" * 60)


❓ What are the characteristics of Chardonnay wine?
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
💬 Chardonnay has buttery, oaky characteristics, with flavors of apple and citrus in cool climates, and tropical fruit notes in warm climates.
------------------------------------------------------------

❓ What wine pairs well with steak?
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
💬 Cabernet Sauvignon, a full-bodied red wine, pairs well with steak, as it is a red meat.
------------------------------------------------------------

❓ How does Databricks detect model drift?
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved 

## 🧹 실습 후 정리 (비용 관리!)

In [0]:
# Vector Search 인덱스 삭제
try:
    vsc.delete_index(VS_ENDPOINT, INDEX_NAME)
    print(f"✅ Vector Search 인덱스 삭제 완료: {INDEX_NAME}")
except Exception as e:
    print(f"⚠️ 삭제 시 오류: {e}")

# 임시 테이블 삭제
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.wine_reviews_lab")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.wine_knowledge_lab")
print("✅ 임시 테이블 삭제 완료")

✅ Vector Search 인덱스 삭제 완료: 3dt005_databricks.wine.wine_knowledge_index
✅ 임시 테이블 삭제 완료


## 🎯 핵심 정리

| 기능 | Databricks 도구 | 특징 |
|------|----------------|------|
| **LLM 호출** | Foundation Model API | 토큰당 과금, 배포 불필요 |
| **SQL AI** | `ai_query()` | SQL만으로 감성분석, 요약, 분류 |
| **문서 검색** | Vector Search | 임베딩 기반 유사도 검색 |
| **RAG** | Vector Search + LLM | 사내 문서 기반 Q&A 구축 |

💡 **Databricks 향기**: `ai_query()`는 SQL Analyst도 LLM을 쓸 수 있게 하는 킬러 기능!

💰 **오늘 LLM 실습 비용**: 1인당 ~$0.20 (max_tokens 제한 덕분)